In [ ]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import pytz
from pathlib import Path

# 1) Caminhos (assumindo que o notebook está em notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

# 2) Conecta ao terminal MT5
if not mt5.initialize():
    print("Erro ao iniciar MT5:", mt5.last_error())
    mt5.shutdown()
    quit()
print("MT5 conectado:", mt5.terminal_info().name)

# 3) Defina o ativo e o timeframe
ATIVO = "PETR4"  # ou "WIN$" para mini-índice, "VALE3", etc.
TIMEFRAME = mt5.TIMEFRAME_M5  # M5 = 5 minutos

# 4) Defina o período em UTC (IMPORTANTE: MT5 armazena em UTC)
timezone = pytz.timezone("Etc/UTC")
data_inicio = datetime(2024, 1, 1, tzinfo=timezone)
data_fim = datetime(2024, 12, 31, tzinfo=timezone)

# 5) Baixa os candles M5
rates = mt5.copy_rates_range(ATIVO, TIMEFRAME, data_inicio, data_fim)

# 6) Verifica se veio algo
if rates is None or len(rates) == 0:
    print(f"Nenhum dado retornado para {ATIVO}.")
    mt5.shutdown()
    quit()

# 7) Converte para DataFrame
df_m5 = pd.DataFrame(rates)

# 8) Converte 'time' de epoch (segundos) para datetime
df_m5['time'] = pd.to_datetime(df_m5['time'], unit='s')

# 9) Salva
output_path = RAW / f'{ATIVO}_M5_2024.parquet'
df_m5.to_parquet(output_path, index=False)

# 10) Desconecta
mt5.shutdown()

print(f"✅ Salvo: {output_path}")
print(f"   Shape: {df_m5.shape}")
print(f"   Período: {df_m5['time'].min()} → {df_m5['time'].max()}")
print(f"\nPrimeiras linhas:")
print(df_m5.head())

MT5 conectado: MetaTrader 5
Nenhum dado retornado para PETR4.


KeyError: 'time'

: 

In [ ]:
import MetaTrader5 as mt5

if not mt5.initialize():
    print("Erro init:", mt5.last_error())
    quit()

# 1) O símbolo PETR4 existe no seu MT5?
info = mt5.symbol_info("PETR4")
print("PETR4 existe?", info is not None)
if info:
    print(f"  Nome: {info.name}")
    print(f"  Visível no Market Watch? {info.visible}")
    print(f"  Trade mode: {info.trade_mode}")

# 2) Busca variações do símbolo (PETR4, PETR4F, PETR4.SA, etc.)
simbolos = mt5.symbols_get()
petr_variacoes = [s.name for s in simbolos if 'PETR' in s.name.upper()]
print(f"\nSímbolos com 'PETR': {petr_variacoes[:20]}")

# 3) Quantos símbolos tem no total?
print(f"\nTotal de símbolos no MT5: {len(simbolos)}")

# 4) Testa com um símbolo que SABEMOS que existe em qualquer MT5 (forex)
rates_test = mt5.copy_rates_range("EURUSD", mt5.TIMEFRAME_M5,
                                    __import__('datetime').datetime(2024, 1, 1),
                                    __import__('datetime').datetime(2024, 1, 31))
print(f"\nEURUSD M5 janeiro/2024: {len(rates_test) if rates_test is not None else 'None'} candles")

mt5.shutdown()

In [1]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import pytz
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

if not mt5.initialize():
    print("Erro:", mt5.last_error())
    quit()

# Usa um símbolo QUE EXISTE na sua conta
ATIVO = "AMD"  # ou "NVDA", "MSFT", "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5

# Garante que está no Market Watch
info = mt5.symbol_info(ATIVO)
if info is None:
    print(f"{ATIVO} não existe no seu MT5")
    mt5.shutdown()
    quit()

if not info.visible:
    mt5.symbol_select(ATIVO, True)
    print(f"{ATIVO} adicionado ao Market Watch")

# Período (UTC)
tz = pytz.timezone("Etc/UTC")
inicio = datetime(2024, 1, 1, tzinfo=tz)
fim = datetime(2024, 12, 31, tzinfo=tz)

# Baixa
rates = mt5.copy_rates_range(ATIVO, TIMEFRAME, inicio, fim)
mt5.shutdown()

# Valida ANTES de criar df (evita KeyError)
if rates is None or len(rates) == 0:
    print(f"❌ Sem dados para {ATIVO} em 2024")
else:
    df_m5 = pd.DataFrame(rates)
    df_m5['time'] = pd.to_datetime(df_m5['time'], unit='s')
    output = RAW / f'{ATIVO}_M5_2024.parquet'
    df_m5.to_parquet(output, index=False)
    print(f"✅ Salvo: {output}")
    print(f"   Shape: {df_m5.shape}")
    print(f"   Período: {df_m5['time'].min()} → {df_m5['time'].max()}")
    print(df_m5.head())

❌ Sem dados para AMD em 2024


In [2]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import pytz
from pathlib import Path

# Caminhos
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

# Conecta
if not mt5.initialize():
    print("Erro:", mt5.last_error())
    quit()

ATIVO = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5

# Garante visibilidade no Market Watch
info = mt5.symbol_info(ATIVO)
if info is None:
    print(f"❌ {ATIVO} não existe no seu MT5")
    mt5.shutdown()
    quit()

if not info.visible:
    mt5.symbol_select(ATIVO, True)
    print(f"✅ {ATIVO} adicionado ao Market Watch")

# --- TESTE 1: cópia por posição (últimos N candles) ---
rates_pos = mt5.copy_rates_from_pos(ATIVO, TIMEFRAME, 0, 10000)
print(f"\n[T1] Últimos 10.000 candles M5: {len(rates_pos) if rates_pos is not None else 'None'}")

# --- TESTE 2: range de datas (2024) ---
tz = pytz.timezone("Etc/UTC")
inicio = datetime(2024, 1, 1, tzinfo=tz)
fim = datetime(2024, 12, 31, tzinfo=tz)

rates_range = mt5.copy_rates_range(ATIVO, TIMEFRAME, inicio, fim)
print(f"[T2] Range 2024 completo: {len(rates_range) if rates_range is not None else 'None'}")

# --- TESTE 3: range mais curto (janeiro 2024) ---
rates_jan = mt5.copy_rates_range(
    ATIVO, TIMEFRAME,
    datetime(2024, 1, 1, tzinfo=tz),
    datetime(2024, 1, 31, tzinfo=tz)
)
print(f"[T3] Range janeiro/2024: {len(rates_jan) if rates_jan is not None else 'None'}")

# --- TESTE 4: qual o candle mais antigo disponível? ---
# Pega o primeiro candle disponível puxando um range muito amplo
rates_full = mt5.copy_rates_range(
    ATIVO, TIMEFRAME,
    datetime(2015, 1, 1, tzinfo=tz),
    datetime(2025, 1, 1, tzinfo=tz)
)
if rates_full is not None and len(rates_full) > 0:
    df_full = pd.DataFrame(rates_full)
    df_full['time'] = pd.to_datetime(df_full['time'], unit='s')
    print(f"\n[T4] Total disponível: {len(df_full)} candles")
    print(f"     Período: {df_full['time'].min()} → {df_full['time'].max()}")
else:
    print("\n[T4] Nenhum candle retornado")

mt5.shutdown()


[T1] Últimos 10.000 candles M5: 10000
[T2] Range 2024 completo: None
[T3] Range janeiro/2024: 1

[T4] Nenhum candle retornado


True

In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

if not mt5.initialize():
    print("Erro:", mt5.last_error())
    quit()

ATIVO = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5

# Baixa os 10.000 candles mais recentes
rates = mt5.copy_rates_from_pos(ATIVO, TIMEFRAME, 0, 10000)
mt5.shutdown()

if rates is None or len(rates) == 0:
    print("❌ Sem dados")
else:
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    
    output = RAW / f'{ATIVO}_M5_recente.parquet'
    df.to_parquet(output, index=False)
    
    print(f"✅ Salvo: {output}")
    print(f"   Shape: {df.shape}")
    print(f"   Período: {df['time'].min()} → {df['time'].max()}")
    print(f"   Dias úteis: {df['time'].dt.date.nunique()}")
    print(f"\nColunas: {df.columns.tolist()}")
    print(f"\nPrimeiras linhas:")
    print(df.head())
    print(f"\nÚltimas linhas:")
    print(df.tail())

✅ Salvo: c:\Users\rondy\Desenvolvimento\b3-ml\data\raw\EURUSD_M5_recente.parquet
   Shape: (10000, 8)
   Período: 2026-08-10 06:40:00 → 2026-09-25 23:55:00
   Dias úteis: 35

Colunas: ['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume']

Primeiras linhas:
                 time     open     high      low    close  tick_volume  \
0 2026-08-10 06:40:00  1.15496  1.15505  1.15488  1.15495          463   
1 2026-08-10 06:45:00  1.15495  1.15508  1.15493  1.15505          421   
2 2026-08-10 06:50:00  1.15504  1.15506  1.15493  1.15499          391   
3 2026-08-10 06:55:00  1.15499  1.15501  1.15488  1.15492          480   
4 2026-08-10 07:00:00  1.15492  1.15498  1.15488  1.15495          413   

   spread  real_volume  
0       0            0  
1       0            0  
2       0            0  
3       0            0  
4       0            0  

Últimas linhas:
                    time     open     high      low    close  tick_volume  \
9995 2026-09-25 23:35:00  1